# Yönetmelik Asistanı · Colab çalıştırıcı

Bu notebook yalnızca ince bir çalıştırıcıdır (plan 12.2); kodun tamamı depodadır.
VS Code'da Google Colab uzantısıyla açın ve çalışma zamanı olarak **T4 GPU** seçin.

**Faz 0 hedefi:** ortamı kurmak ve `smoke` sağlık kontrolünü geçmek.

## 1 · GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 2 · Drive ve önbellek dizinleri

İndeks ve veriler her zaman Drive'da tutulur (oturum düşerse kaybolmasın diye).
Model dosyalarını Drive'a yazmak isteğe bağlıdır: ücretsiz Drive kotası 15 GB'tır ve
Qwen3.5-9B (~6,6 GB) ile BGE-M3 ve reranker (~4,5 GB) birlikte bunun büyük kısmını doldurur.
Varsayılan olarak modeller her oturumda yeniden indirilir.

**VS Code'da:** Drive'ı Colab sunucu menüsündeki *Mount Google Drive to server* ile bağlayın,
sonra bu hücreyi çalıştırın. Drive bağlı değilse hücre durur; böylece veriler yanlışlıkla
sunucunun geçici diskine yazılmaz.

In [ ]:
import os

DRIVE = '/content/drive/MyDrive'
if not os.path.isdir(DRIVE):
    try:  # Colab web arayüzünde çalışır; VS Code'da sunucu menüsünü kullanın
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'drive.mount çalışmadı: {e}')
if not os.path.isdir(DRIVE):
    raise RuntimeError(
        "Google Drive bağlı değil. VS Code'da Colab sunucu menüsünden "
        "'Mount Google Drive to server' seçin ve bu hücreyi tekrar çalıştırın."
    )

BASE = '/content/drive/MyDrive/yonetmelik-asistani'
MODELLERI_DRIVEDA_TUT = False  # True: indirme süresi kısalır, Drive kotası dolar

os.environ['YA_DATA_DIR'] = f'{BASE}/data'
os.makedirs(os.environ['YA_DATA_DIR'], exist_ok=True)
if MODELLERI_DRIVEDA_TUT:
    os.environ['HF_HOME'] = f'{BASE}/hf-cache'
    os.environ['OLLAMA_MODELS'] = f'{BASE}/ollama-models'
print({k: os.environ.get(k) for k in ('YA_DATA_DIR', 'HF_HOME', 'OLLAMA_MODELS')})

## 3 · Depoyu al ve paketi kur

Depo herkese açık olduğu için token gerekmez. Depo özel yapılırsa Colab'ın
*Secrets* bölümüne `GITHUB_TOKEN` ekleyin.

In [ ]:
REPO_URL = 'https://github.com/Cnrygt11/yonetmelik-asistani.git'
REPO_DIR = '/content/yonetmelik-asistani'

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        REPO_URL = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%pip install -q -e "{REPO_DIR}[ingest,search,dense]"

## 4 · Ollama'yı kur, başlat ve modeli indir

In [ ]:
MODEL = 'qwen3.5:9b'  # yedek: 'gemma4:e4b'
!bash {REPO_DIR}/scripts/colab_setup.sh {MODEL}

## 5 · Sağlık kontrolü

Beş soru çalışır: dördü Türkçe akıcılık ve hız, biri boş bağlamda ret davranışı.
Rapor oturum kapanınca kaybolmasın diye Drive'a yazılır. Faz 0 çıkışında bu raporu
depodaki `eval/reports/` klasörüne ekleyin.

In [ ]:
from datetime import datetime
os.makedirs(f'{BASE}/reports', exist_ok=True)
REPORT = f"{BASE}/reports/smoke_{datetime.now():%Y%m%d_%H%M%S}.json"
!cd {REPO_DIR} && python -m yonetmelik_asistani.smoke --model {MODEL} --report {REPORT}
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6 · Faz 2: derlem, indeks ve arama değerlendirmesi

Bu bölüm **Ollama gerektirmez**; 4 ve 5 atlanabilir (1-3 çalıştırılmış olmalı).
Belgeler Drive'da (`YA_DATA_DIR`), indeks Colab'ın yerel diskinde tutulur: Qdrant'ın SQLite
dosyası Drive'da kilitlenebiliyor. İndeksin yeniden kurulması yaklaşık bir dakika sürer.

**6.1** Drive'daki `data/raw` belgelerinden metni çıkar ve doğrula. Colab internete çıkmaz:
`.gov.tr` siteleri Colab'dan erişilemiyor. İndirme ve çapraz kontrol yerel bilgisayarda yapılır,
güncel `data/raw` klasörü Drive'a yüklenir. `extract` her dosyanın sha256 özetini
`configs/corpus_lock.json` ile karşılaştırır.

In [ ]:
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
!cd {REPO_DIR} && python -m yonetmelik_asistani.ingest extract

**6.2** Parçala, BM25 ve BGE-M3 indekslerini kur (BGE-M3 ilk seferde ~2,3 GB indirir).

In [ ]:
!cd {REPO_DIR} && python -m yonetmelik_asistani.chunking && python -m yonetmelik_asistani.index

**6.3** Geliştirme setinde dört arama kipini karşılaştır (reranker ilk seferde ~2,3 GB indirir).
Çıktının tamamını paylaşın; rapor Drive'a da kopyalanır.

In [ ]:
!cd {REPO_DIR} && python eval/run_eval.py --methods bm25,dense,hybrid,hybrid_rerank --tokenizers char --sozluk ikisi
!mkdir -p {BASE}/reports && cp {REPO_DIR}/eval/reports/retrieval_*.json {BASE}/reports/
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

**6.4** Örnek arama (her aşamanın sırası ve skoru görünür).

In [ ]:
!cd {REPO_DIR} && python -m yonetmelik_asistani.retrieval "Kaydımı dondurmak istiyorum, ne zamana kadar başvurmalıyım?" -k 3

## 7 · Uçtan uca deneme (Faz 4 öncesi)

Aynı T4'te Ollama (Qwen3.5-9B) + BGE-M3 + reranker birlikte çalışır ve 6 soruya kaynak gösteren
cevap üretilir. Amaç ölçmek değil; bellek, süre, kaynak etiketleri ve ret davranışını erken görmek.

**Önce:** 1-3, ardından **4** (Ollama kurulumu ve model indirme, ~5 dk) çalıştırılmış olmalı.
İndeks yoksa 6.1 ve 6.2 de gerekir. Rapor Drive'a yazılır (`reports/e2e_*.json`).

In [ ]:
from datetime import datetime
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
if not os.path.isdir('/content/ya-index/qdrant'):
    print('İndeks yok: önce 6.1 ve 6.2 çalıştırılmalı')
else:
    E2E = f"{BASE}/reports/e2e_{datetime.now():%Y%m%d_%H%M%S}.json"
    !cd {REPO_DIR} && python eval/e2e_deneme.py --report {E2E}